In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 05_facts/03_fact_medications
# Purpose  : fact_medications - Grain: One row = one prescription medication record.
#
# Post-Review Refinements:
#   1) medication_code/description decoupled into a standalone dim_medication
#      (following the exact pattern of dim_condition/dim_procedure/dim_observation_type)—
#      eliminating text repetition across all 4.2 million medication records.
#   2) treatment_duration_days: After confirming that applying current_date() for active
#      prescriptions (stop_date IS NULL) freezes a static snapshot timestamp at row insertion time
#      (since this Fact table is Streaming/Append-only and does not recompute automatically on subsequent days),
#      we reset it to NULL for active prescriptions. Duration is calculated exclusively for completed
#      prescriptions (stop_date IS NOT NULL) to maintain historical accuracy and eliminate misleading metrics.
#
# Core Business Rules:
#   - is_active and stop_date passed directly from Silver as-is.
#   - reason_key: Enforces the Unknown Member pattern (-1), aligned with fact_encounters.
#   - Records with STOP < START were validated and routed to Quarantine at the Silver layer.
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.table(
    name="covid19_lakehouse.gold.fact_medications",
    comment="Fact table - one row per medication prescription record",
    cluster_by=["patient_key", "start_date_key"],
)
def fact_medications():
    medications = dlt.read_stream("covid19_lakehouse.silver.medications")

    patients = (
        spark.read.table("covid19_lakehouse.gold.dim_patient")
        .filter(F.col("is_current") == True)  # noqa: E712
        .select(F.col("patient_id"), F.col("patient_key"))
    )
    payers = spark.read.table("covid19_lakehouse.gold.dim_payer").select(
        F.col("payer_id"), F.col("payer_key")
    )
    reasons = spark.read.table("covid19_lakehouse.gold.dim_reason").select(
        F.col("reason_code"), F.col("reason_key")
    )
    medication_dim = spark.read.table("covid19_lakehouse.gold.dim_medication").select(
        F.col("medication_code"), F.col("medication_key")
    )

    df = (
        medications
        .join(patients, on="patient_id", how="left")
        .join(payers, on="payer_id", how="left")
        .join(reasons, on="reason_code", how="left")
        .join(medication_dim, on="medication_code", how="left")
    )

    return df.select(
        F.xxhash64("patient_id", "encounter_id", "medication_code", "start_date").alias(
            "medication_fact_key"
        ),
        F.col("patient_key"),
        F.col("encounter_id"),  # Degenerate Dimension
        F.col("payer_key"),
        F.col("medication_key"),
        F.coalesce(F.col("reason_key"), F.lit(-1)).alias("reason_key"),
        F.date_format(F.col("start_date"), "yyyyMMdd").cast("int").alias("start_date_key"),
        F.date_format(F.col("stop_date"), "yyyyMMdd").cast("int").alias("stop_date_key"),
        F.col("dispenses"),
        F.col("base_cost"),
        F.col("total_cost"),
        F.col("payer_coverage"),
        (F.col("total_cost") - F.col("payer_coverage")).alias("patient_out_of_pocket"),
        F.when(
            F.col("stop_date").isNotNull(),
            F.datediff(F.col("stop_date"), F.col("start_date")),
        ).alias("treatment_duration_days"),
        F.col("is_active"),
    )